# "rs-client-libraries" flow example with Prefect (not Dask)

See the associated:

  * Python module: [rs_client_flow.py](./rs_client_flow.py)
  * YAML file: [rs_client_flow.yaml](./rs_client_flow.yaml)

## Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *  

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000


In [3]:
# Other imports
import os
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

In [4]:
# Set env vars for the prefect flow
os.environ["RS_SERVER_STAGING_ADDRESS"] = \
    os.environ["RSPY_WEBSITE"] if cluster_mode else \
    os.environ["RSPY_HOST_STAGING"]
os.environ["RS_API_KEY"] = "TO_BE_DEFINED" # should be passed as a prefect block ?
os.environ["RS_OWNER"] = OWNER_ID # jupyter username

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [5]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{share_bucket.bucket_name}/{share_bucket.bucket_folder}/{s3_code_folder}'")

# Upload local directory and resources contents
await share_bucket.put_directory(local_path = ".", to_path = s3_code_folder)
await share_bucket.put_directory(local_path = "../../resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{share_bucket.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/code'


In [6]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./rs_client_flow.yaml"

10:25:13.760 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.2"
10:25:13.814 | WARNING | prefect.utilities.templating - Value for placeholder 'PREFECT_SHARE_BUCKET' not found in provided values. Please ensure that the placeholder is spelled correctly and that the corresponding value is provided.


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'get-staging-jobs/sprint19-rs-client' successfully created with   │
│ id '4f5dd369-7294-478d-867c-9085fb8a6e19'.                                   │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/4f5dd369-7294-478d-867c-9085fb8a6e19


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'get-staging-jobs/sprint19-rs-client'



In [7]:
deploy_name = "get-staging-jobs/sprint19-rs-client"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'get-staging-jobs/sprint19-rs-client'


## Run Prefect flow

In [8]:
%%bash -s "$deploy_name"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --watch

Creating flow run for deployment 'get-staging-jobs/sprint19-rs-client'...
Created flow run 'ivory-wallaby'.
└── UUID: c524d8b7-acf6-4f49-915d-7553c0b4580b
└── Parameters: {}
└── Job Variables: {}
└── Scheduled start time: 2025-05-22 10:25:18 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/c524d8b7-acf6-4f49-915d-7553c0b4580b
Watching flow run 'ivory-wallaby'...


10:25:18.751 | INFO    | prefect - Flow run is in state 'Scheduled'
10:25:26.841 | INFO    | prefect - Flow run is in state 'Pending'
10:25:30.200 | INFO    | prefect - Flow run is in state 'Running'
10:25:30.378 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


NOTE: we could also call the Prefect flow from Python code. This is useful to debug.

In [9]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import rs_client_flow
    reload(rs_client_flow)
    
    # Run the flow
    results = rs_client_flow.get_staging_jobs()
    display(results)